In [1]:
# Re-run after reset: execute the full script again to produce outputs.
import numpy as np

np.set_printoptions(precision=6, suppress=True)

def make_tall_matrix_with_singular_values(m=3, n=2, svals=(1.0, 1e-6), seed=0):
    rng = np.random.default_rng(seed)
    Qm, _ = np.linalg.qr(rng.standard_normal((m, m)))
    U = Qm[:, :n]
    Qn, _ = np.linalg.qr(rng.standard_normal((n, n)))
    V = Qn
    S = np.diag(svals)
    A = U @ S @ V.T
    return A, U, S, V

def cond2(A):
    s = np.linalg.svd(A, compute_uv=False)
    return s.max()/s.min()

def robust_ls_qr(A, y, rcond=None):
    x, residuals, rank, s = np.linalg.lstsq(A, y, rcond=rcond)
    resnorm = np.linalg.norm(A @ x - y)
    return x, resnorm

def normal_equations(A, y):
    J = A.T @ A
    b = A.T @ y
    x = np.linalg.solve(J, b)
    resnorm = np.linalg.norm(A @ x - y)
    return x, resnorm

def run_case(svals=(1.0, 1e-6), noise=0.0, seed=123, print_matrices=False):
    A, U, S, V = make_tall_matrix_with_singular_values(3, 2, svals=svals, seed=seed)
    x_star = np.array([[1.0], [1.0]])
    y = A @ x_star

    if noise > 0.0:
        rng = np.random.default_rng(seed+999)
        A = A + noise * rng.standard_normal(A.shape)
        y = y + noise * rng.standard_normal(y.shape)

    x_ls, rn_ls = robust_ls_qr(A, y)
    x_ne, rn_ne = normal_equations(A, y)

    kA = cond2(A)
    J = A.T @ A
    kJ = cond2(J)
    rel_err_ls = np.linalg.norm(x_ls - x_star) / np.linalg.norm(x_star)
    rel_err_ne = np.linalg.norm(x_ne - x_star) / np.linalg.norm(x_star)

    print("="*80)
    print(f"svals={svals}, noise={noise}, seed={seed}")
    print(f"cond2(A) = {kA:.3e} ; cond2(J) = {kJ:.3e}  (≈ cond2(A)^2)")
    if print_matrices:
        print("\nA =\n", A)
        print("\nJ = A^T A =\n", J)

    print("\nx*   =\n", x_star.ravel())
    print("x_ls (QR/SVD) =\n", x_ls.ravel())
    print("x_ne (normal equations) =\n", x_ne.ravel())

    print(f"\n||Ax_ls - y||_2 = {rn_ls:.3e} ; ||Ax_ne - y||_2 = {rn_ne:.3e}")
    print(f"rel_err (QR/SVD)  = {rel_err_ls:.3e}")
    print(f"rel_err (Normal)  = {rel_err_ne:.3e}")
    print("="*80)
    print()

cases = [
    {"svals": (1.0, 1e-6), "noise": 0.0,    "seed": 43, "print_matrices": True},
    {"svals": (1.0, 1e-6), "noise": 1e-12,  "seed": 44, "print_matrices": False},
    {"svals": (1.0, 1e-8), "noise": 0.0,    "seed": 45, "print_matrices": False},
    {"svals": (1.0, 1e-8), "noise": 1e-12,  "seed": 46, "print_matrices": False},
]

for cfg in cases:
    run_case(**cfg)


svals=(1.0, 1e-06), noise=0.0, seed=43
cond2(A) = 1.000e+06 ; cond2(J) = 1.000e+12  (≈ cond2(A)^2)

A =
 [[ 0.258252 -0.025652]
 [-0.960847  0.095437]
 [ 0.017614 -0.00175 ]]

J = A^T A =
 [[ 0.990231 -0.098356]
 [-0.098356  0.009769]]

x*   =
 [1. 1.]
x_ls (QR/SVD) =
 [1. 1.]
x_ne (normal equations) =
 [0.999998 0.999983]

||Ax_ls - y||_2 = 3.711e-16 ; ||Ax_ne - y||_2 = 1.666e-11
rel_err (QR/SVD)  = 1.735e-10
rel_err (Normal)  = 1.178e-05

svals=(1.0, 1e-06), noise=1e-12, seed=44
cond2(A) = 1.000e+06 ; cond2(J) = 1.000e+12  (≈ cond2(A)^2)

x*   =
 [1. 1.]
x_ls (QR/SVD) =
 [1. 1.]
x_ne (normal equations) =
 [0.999964 0.999912]

||Ax_ls - y||_2 = 2.735e-14 ; ||Ax_ne - y||_2 = 9.474e-11
rel_err (QR/SVD)  = 3.127e-07
rel_err (Normal)  = 6.730e-05

svals=(1.0, 1e-08), noise=0.0, seed=45
cond2(A) = 1.000e+08 ; cond2(J) = 6.800e+15  (≈ cond2(A)^2)

x*   =
 [1. 1.]
x_ls (QR/SVD) =
 [1. 1.]
x_ne (normal equations) =
 [0.562454 0.658928]

||Ax_ls - y||_2 = 6.206e-17 ; ||Ax_ne - y||_2 = 5.548e-0

In [4]:
import numpy as np
np.set_printoptions(precision=18, suppress=False)

def make_A_eps(eps: float):
    return np.array([[1.0, 0.0],
                     [0.0, eps],
                     [0.0, 0.0]], dtype=float)

def cond2_numeric(A):
    s = np.linalg.svd(A, compute_uv=False)
    # return np.inf if np.isclose(s.min(), 0.0) else s.max()/s.min()
    return s.max()/s.min()

def robust_ls_qr(A, y, rcond=None):
    x, residuals, rank, s = np.linalg.lstsq(A, y, rcond=rcond)
    return x, np.linalg.norm(A @ x - y)

def normal_equations(A, y):
    J = A.T @ A
    b = A.T @ y
    x = np.linalg.solve(J, b)
    return x, np.linalg.norm(A @ x - y)

def run_case_eps(eps: float, noise: float = 0.0, seed: int = 0, show_mats: bool = True):
    rng = np.random.default_rng(seed)
    A = make_A_eps(eps)
    x_star = np.array([[1.0], [1.0]])
    y = A @ x_star

    if noise > 0.0:
        A = A + noise * rng.standard_normal(A.shape)
        y = y + noise * rng.standard_normal(y.shape)

    x_ls, rn_ls = robust_ls_qr(A, y)
    x_ne, rn_ne = normal_equations(A, y)

    kA_num = cond2_numeric(A)
    J = A.T @ A
    kJ_num = cond2_numeric(J)

    print("="*100)
    print(f"eps={eps:.0e}, noise={noise:.0e}, seed={seed}")
    if show_mats:
        print("\nA =\n", A)
        print("\nJ = A^T A =\n", J)

    if noise == 0.0:
        print(f"\nAnalytical: cond2(A)=1/eps={1/eps:.3e} ; cond2(J)=1/eps^2={(1/eps**2):.3e}")
    print(f"Numeric:    cond2(A)={kA_num:.3e} ; cond2(J)={kJ_num:.3e}")

    print("\nx*   =\n", x_star.ravel())
    print("x_ls (QR/SVD) =\n", x_ls.ravel())
    print("x_ne (normal equations) =\n", x_ne.ravel())

    print(f"\n||Ax_ls - y||_2 = {rn_ls:.3e} ; ||Ax_ne - y||_2 = {rn_ne:.3e}")
    print(f"rel_err (QR/SVD)  = {np.linalg.norm(x_ls - x_star)/np.linalg.norm(x_star):.3e}")
    print(f"rel_err (Normal)  = {np.linalg.norm(x_ne - x_star)/np.linalg.norm(x_star):.3e}")
    print("="*100+"\n")

for eps in (1e-6, 1e-9):
    run_case_eps(eps, noise=0.0,  seed=0, show_mats=True)
    run_case_eps(eps, noise=1e-12, seed=1, show_mats=False)


eps=1e-06, noise=0e+00, seed=0

A =
 [[1.e+00 0.e+00]
 [0.e+00 1.e-06]
 [0.e+00 0.e+00]]

J = A^T A =
 [[1.e+00 0.e+00]
 [0.e+00 1.e-12]]

Analytical: cond2(A)=1/eps=1.000e+06 ; cond2(J)=1/eps^2=1.000e+12
Numeric:    cond2(A)=1.000e+06 ; cond2(J)=1.000e+12

x*   =
 [1. 1.]
x_ls (QR/SVD) =
 [1. 1.]
x_ne (normal equations) =
 [1. 1.]

||Ax_ls - y||_2 = 0.000e+00 ; ||Ax_ne - y||_2 = 0.000e+00
rel_err (QR/SVD)  = 0.000e+00
rel_err (Normal)  = 0.000e+00

eps=1e-06, noise=1e-12, seed=1
Numeric:    cond2(A)=1.000e+06 ; cond2(J)=1.000e+12

x*   =
 [1. 1.]
x_ls (QR/SVD) =
 [0.999999999998296  1.0000015538398441]
x_ne (normal equations) =
 [0.999999999998296  1.0000015538398437]

||Ax_ls - y||_2 = 9.872e-13 ; ||Ax_ne - y||_2 = 9.872e-13
rel_err (QR/SVD)  = 1.099e-06
rel_err (Normal)  = 1.099e-06

eps=1e-09, noise=0e+00, seed=0

A =
 [[1.e+00 0.e+00]
 [0.e+00 1.e-09]
 [0.e+00 0.e+00]]

J = A^T A =
 [[1.e+00 0.e+00]
 [0.e+00 1.e-18]]

Analytical: cond2(A)=1/eps=1.000e+09 ; cond2(J)=1/eps^2=1.000e+

In [7]:
# Rotated version of the user's diagonal example:
# A0 = [[1,0],[0,eps],[0,0]], A = Q @ A0 with random orthogonal Q (3x3).
# Compare QR/SVD least-squares vs. Normal Equations across eps, seeds, and small noise.

import numpy as np

np.set_printoptions(precision=6, suppress=True)

def make_A_rotated(eps: float, seed: int):
    rng = np.random.default_rng(seed)
    # base A0
    A0 = np.array([[1.0, 0.0],
                   [0.0, eps],
                   [0.0, 0.0]], dtype=float)
    # random orthogonal Q via QR
    Q, _ = np.linalg.qr(rng.standard_normal((3, 3)))
    A = Q @ A0
    return A, Q

def cond2(A):
    s = np.linalg.svd(A, compute_uv=False)
    return np.inf if np.isclose(s.min(), 0.0) else s.max()/s.min()

def ls_qr(A, y, rcond=None):
    x, *_ = np.linalg.lstsq(A, y, rcond=rcond)
    return x, np.linalg.norm(A @ x - y)

def normal_eq(A, y):
    J = A.T @ A
    b = A.T @ y
    x = np.linalg.solve(J, b)
    return x, np.linalg.norm(A @ x - y)

def run_suite(eps_list=(1e-6, 1e-11), seeds=(0,1,2), noise_levels=(0.0, 1e-12)):
    for eps in eps_list:
        for seed in seeds:
            A, Q = make_A_rotated(eps, seed)
            x_star = np.array([[1.0],[1.0]])
            y_clean = A @ x_star
            for noise in noise_levels:
                rng = np.random.default_rng(10_000 + seed)
                A_use = A.copy()
                y = y_clean.copy()
                if noise > 0.0:
                    A_use += noise * rng.standard_normal(A.shape)
                    y += noise * rng.standard_normal(y.shape)

                x_ls, rn_ls = ls_qr(A_use, y)
                x_ne, rn_ne = normal_eq(A_use, y)

                kA = cond2(A_use)
                kJ = cond2(A_use.T @ A_use)
                rel_ls = np.linalg.norm(x_ls - x_star)/np.linalg.norm(x_star)
                rel_ne = np.linalg.norm(x_ne - x_star)/np.linalg.norm(x_star)

                print("="*110)
                print(f"eps={eps:.0e}, seed={seed}, noise={noise:.0e}")
                print(f"cond2(A)={kA:.3e} ; cond2(J)={kJ:.3e} (≈ cond2(A)^2)")
                print("x*   =", x_star.ravel())
                print("x_ls =", x_ls.ravel())
                print("x_ne =", x_ne.ravel())
                print(f"||Ax_ls - y||={rn_ls:.3e} ; ||Ax_ne - y||={rn_ne:.3e}")
                print(f"rel_err (QR/SVD)={rel_ls:.3e} ; rel_err (Normal)={rel_ne:.3e}")
                print("="*110)
                print()

run_suite()


eps=1e-06, seed=0, noise=0e+00
cond2(A)=1.000e+06 ; cond2(J)=inf (≈ cond2(A)^2)
x*   = [1. 1.]
x_ls = [1. 1.]
x_ne = [1. 1.]
||Ax_ls - y||=7.076e-17 ; ||Ax_ne - y||=0.000e+00
rel_err (QR/SVD)=4.836e-11 ; rel_err (Normal)=1.349e-12

eps=1e-06, seed=0, noise=1e-12
cond2(A)=1.000e+06 ; cond2(J)=inf (≈ cond2(A)^2)
x*   = [1. 1.]
x_ls = [1. 1.]
x_ne = [1. 1.]
||Ax_ls - y||=1.053e-12 ; ||Ax_ne - y||=1.053e-12
rel_err (QR/SVD)=5.336e-08 ; rel_err (Normal)=5.331e-08

eps=1e-06, seed=1, noise=0e+00
cond2(A)=1.000e+06 ; cond2(J)=inf (≈ cond2(A)^2)
x*   = [1. 1.]
x_ls = [1. 1.]
x_ne = [1. 1.]
||Ax_ls - y||=1.755e-16 ; ||Ax_ne - y||=2.355e-16
rel_err (QR/SVD)=1.282e-10 ; rel_err (Normal)=1.645e-11

eps=1e-06, seed=1, noise=1e-12
cond2(A)=1.000e+06 ; cond2(J)=inf (≈ cond2(A)^2)
x*   = [1. 1.]
x_ls = [1. 1.]
x_ne = [1. 1.]
||Ax_ls - y||=1.644e-12 ; ||Ax_ne - y||=1.644e-12
rel_err (QR/SVD)=2.958e-07 ; rel_err (Normal)=2.957e-07

eps=1e-06, seed=2, noise=0e+00
cond2(A)=1.000e+06 ; cond2(J)=inf (≈ cond